Multi asset stat arb with option volatility & PCA/K means
To do:
1. Finish Securities Version
-->K means what is optimal k & optimize for sharpe in each candidate
-->how do i select which portfolio is best?
-->Johansen cointegration test & Augmented Dickey Fuller
2. Implement Options Version
-->optimize for sortino instead of sharpe
3. Use Sensitivity Analysis to check validity+ PNL analysis with change due to time, change due to market in addition to normal
4. Deploy to alpaca


Areas of improvement from what I read
Improve PCA by sector, condense stocks from indexes to only look at relevant ones
Determine optimal time intervals to recluster
OU disintegration when mispricing grows beyond a certain point
for transaction costs remember 2 layers (fixed per trade and then dynamic based on kyle law per trade)

In [7]:
#Packages
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import wrds
import matplotlib.pyplot as plt

In [20]:
#Data Pipeline
db=wrds.Connection(wrds_username='bosekaikini')
db.create_pgpass_file()
print(db.list_libraries())
#wrds cloud is another option but we will run it locally for now

OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "bosekaikini"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
#Data testing
db.describe_table(library='crsp', table='dsi')

sample_query="""
SELECT date,usdval,ewretx
FROM crsp.dsi
WHERE date >= '2020-01-01'
"""

df=db.raw_sql(sample_query)
print(df)

Approximately 26051 rows in crsp.dsi.
            date         usdval    ewretx
0     2020-01-02  41156335900.0  0.005428
1     2020-01-03  41464750500.0 -0.003072
2     2020-01-06  41211825700.0  0.004519
3     2020-01-07  41345979800.0 -0.000057
4     2020-01-08  41255458200.0  0.000903
...          ...            ...       ...
1253  2024-12-24  74889584800.0   0.01025
1254  2024-12-26  75679041300.0  0.012038
1255  2024-12-27  75702330000.0 -0.004754
1256  2024-12-30  74890031400.0 -0.001173
1257  2024-12-31  74153696300.0 -0.000491

[1258 rows x 3 columns]


In [ ]:
symbols = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'META']
price_query = """
SELECT a.date, b.ticker, a.ret
FROM crsp.dsf AS a
JOIN crsp.stocknames AS b
  ON a.permno = b.permno
 AND a.date BETWEEN b.namedt AND b.nameenddt
WHERE a.date >= '2020-01-01'
  AND b.ticker IN ('AAPL', 'MSFT', 'GOOG', 'AMZN', 'META')
"""

# Pull the ticker-level data into a dedicated dataframe.
df_prices = db.raw_sql(price_query, date_cols=['date'])
df_prices = df_prices.sort_values(['ticker', 'date'])

plt.figure(figsize=(12, 6))
for symbol in symbols:
    s = df_prices[df_prices['ticker'] == symbol].copy()
    # Convert simple returns to cumulative growth of $1 for a clean comparison plot.
    s['cum_return'] = (1 + s['ret'].fillna(0)).cumprod() - 1
    plt.plot(s['date'], s['cum_return'], label=symbol)

plt.title('Cumulative Returns since 2020-01-01')
plt.xlabel('Date')
plt.ylabel('Cumulative Return of $1')
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'db' is not defined

In [ ]:
# Correlation across tickers, using a wide returns table.
returns_wide = df_prices.pivot(index='date', columns='ticker', values='ret')
corr_matrix = returns_wide.corr()
print(corr_matrix)

#p=0.7 threshold
filtered_matrix=corr_matrix[corr_matrix.abs() > 0.7]
print(filtered_matrix)

#assume i have the 2 correlated assets
pair=('AAPL', 'MSFT')
pair_gap=df_prices[df_prices['ticker'] == pair[0]].set_index('date')['ret'] - df_prices[df_prices['ticker'] == pair[1]].set_index('date')['ret']
if pair_gap.mean() > 0:
    short,long=pair[0],pair[1]

NameError: name 'df_prices' is not defined

In [ ]:
# Find hedge ratios and run ADF tests in a vectorized workflow.
from statsmodels.tsa.stattools import adfuller

# Use the ticker x date returns matrix created earlier.
R = returns_wide.sort_index().copy()
R = R.fillna(0.0)

# Hedge ratio beta(i on j) = Cov(i, j) / Var(j) for all ticker pairs at once.
cov_matrix = R.cov()
var_by_ticker = pd.Series(np.diag(cov_matrix.values), index=cov_matrix.index)
hedge_ratio = cov_matrix.div(var_by_ticker, axis=1)
np.fill_diagonal(hedge_ratio.values, np.nan)

print(hedge_ratio.round(4))

# Build all unique ticker pairs (upper triangle) and evaluate ADF on spreads.
tickers = R.columns.to_numpy()
i_idx, j_idx = np.triu_indices(len(tickers), k=1)

adf_rows = []
for i, j in zip(i_idx, j_idx):
    y = tickers[i]
    x = tickers[j]
    beta = hedge_ratio.loc[y, x]
    spread = (R[y] - beta * R[x]).dropna()

    # Skip degenerate spreads that cannot support a meaningful ADF test.
    if spread.nunique() < 3:
        pval = np.nan
        test_stat = np.nan
    else:
        test_stat, pval, *_ = adfuller(spread, autolag='AIC')

    adf_rows.append({
        'y_ticker': y,
        'x_ticker': x,
        'beta': beta,
        'adf_stat': test_stat,
        'adf_pvalue': pval
    })

adf_results = pd.DataFrame(adf_rows).sort_values('adf_pvalue', na_position='last')
print('\nTop candidate pairs by ADF p-value:')
print(adf_results.head(10))

In [ ]:
# now time to use our pca and clustering

# Build a stock-by-time return matrix.
returns_wide = (
    df_prices.pivot(index='date', columns='ticker', values='ret')
    .sort_index()
    .dropna(axis=1, how='all')
    .fillna(0)
)

# Each stock is one observation and each date is a feature.
stock_matrix = returns_wide.T
stock_matrix = stock_matrix.sub(stock_matrix.mean(axis=1), axis=0)
stock_matrix = stock_matrix.div(stock_matrix.std(axis=1).replace(0, np.nan), axis=0).fillna(0)

# Reduce dimensionality before clustering so K-means sees the dominant return structure.
n_components = min(3, stock_matrix.shape[0], stock_matrix.shape[1])
pca = PCA(n_components=n_components, random_state=0)
stock_embeddings = pca.fit_transform(stock_matrix)

# Cluster stocks on the PCA embedding.
n_clusters = min(3, len(stock_matrix.index))
kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto')
stock_clusters = kmeans.fit_predict(stock_embeddings)

cluster_mapping = pd.Series(stock_clusters, index=stock_matrix.index, name='cluster')
print(cluster_mapping.sort_values())

In [ ]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

# Pick one cluster to test for cointegration; change the cluster id if you want a different basket.
selected_cluster = cluster_mapping.value_counts().index[0]
cluster_tickers = cluster_mapping[cluster_mapping == selected_cluster].index.tolist()

if len(cluster_tickers) < 2:
    print('Need at least two stocks in the cluster for Johansen.')
else:
    # Johansen works on price levels, so rebuild a normalized price series from returns.
    cluster_returns = (
        df_prices[df_prices['ticker'].isin(cluster_tickers)]
        .pivot(index='date', columns='ticker', values='ret')
        .sort_index()
        .fillna(0)
    )
    cluster_prices = (1 + cluster_returns).cumprod()

    johansen = coint_johansen(cluster_prices, det_order=0, k_ar_diff=1)
    weights = pd.Series(johansen.evec[:, 0], index=cluster_prices.columns)
    weights = weights / weights.abs().sum()

    long_weights = weights[weights > 0]
    short_weights = weights[weights < 0]

    print('Cluster tickers:', cluster_tickers)
    print('Normalized Johansen weights:')
    print(weights.sort_values())
    print('\nLong leg:')
    print(long_weights.sort_values(ascending=False))
    print('\nShort leg:')
    print(short_weights.sort_values())

In [ ]:
# transaction cost layer

